# Daily Challenge – Breast Cancer Classification

**Dataset :** Breast Cancer Wisconsin (569 patients, 30 features)  
**Objectif :** Prédire si une tumeur est **Maligne (M)** ou **Bénigne (B)** en comparant 4 modèles de classification.

**Modèles :** Logistic Regression · K-Nearest Neighbours · Random Forest · SVM

## Partie 1 – Exploratory Data Analysis

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay
)

%matplotlib inline
sns.set_theme(style='whitegrid')

RANDOM_STATE = 42

In [ ]:
df = pd.read_csv('data.csv')

print('Dimensions :', df.shape)
display(df.head())

In [ ]:
# Vérification des valeurs manquantes
print('Valeurs manquantes par colonne :')
missing = df.isnull().sum()
print(missing[missing > 0])

In [ ]:
# Supprimer les colonnes inutiles :
# - 'id'          : identifiant patient, sans valeur prédictive
# - 'Unnamed: 32' : colonne vide (100% NaN)
df = df.drop(columns=['id', 'Unnamed: 32'])

print('Dimensions après nettoyage :', df.shape)
print('Colonnes restantes :', df.columns.tolist())

In [ ]:
# Countplot de la colonne diagnosis avec la palette magma
plt.figure(figsize=(7, 5))
ax = sns.countplot(x='diagnosis', data=df, palette='magma', order=['B', 'M'])

for bar in ax.patches:
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 5,
        int(bar.get_height()),
        ha='center', fontsize=12, fontweight='bold'
    )

plt.title('Répartition des diagnostics (B = Bénigne, M = Maligne)',
          fontsize=13, fontweight='bold')
plt.xlabel('Diagnostic')
plt.ylabel('Nombre de patients')
plt.xticks([0, 1], ['Bénigne (B)', 'Maligne (M)'])
plt.tight_layout()
plt.show()

In [ ]:
# Distribution de quelques features clés par diagnostic
key_features = ['radius_mean', 'texture_mean', 'area_mean', 'concavity_mean']

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, feat in zip(axes, key_features):
    df[df['diagnosis'] == 'B'][feat].hist(bins=25, ax=ax, alpha=0.6,
                                           color='#3498db', label='B', edgecolor='white')
    df[df['diagnosis'] == 'M'][feat].hist(bins=25, ax=ax, alpha=0.6,
                                           color='#e74c3c', label='M', edgecolor='white')
    ax.set_title(feat, fontsize=9)
    ax.legend(title='Diagnostic')

plt.suptitle('Distribution des features clés par diagnostic',
             fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()

## Partie 2 – Préprocessing, Construction des modèles et Évaluation

In [ ]:
# Nombre de valeurs uniques dans la colonne 'diagnosis'
print('Valeurs uniques dans diagnosis :')
print(df['diagnosis'].value_counts())

In [ ]:
# Encodage : B (Bénigne) → 0 | M (Maligne) → 1
df['diagnosis'] = df['diagnosis'].map({'B': 0, 'M': 1})

print('Après encodage :')
print(df['diagnosis'].value_counts())
display(df[['diagnosis']].head())

In [ ]:
X = df.drop(columns=['diagnosis'])
y = df['diagnosis']

# Standardisation (important pour KNN et SVM)
scaler  = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print(f'Train : {X_train.shape} | Test : {X_test.shape}')
print(f'Proportion de malignes dans le train : {y_train.mean():.2%}')
print(f'Proportion de malignes dans le test  : {y_test.mean():.2%}')

In [ ]:
# Logistic Regression
lr = LogisticRegression(max_iter=10000, random_state=RANDOM_STATE)
lr.fit(X_train, y_train)
acc_lr = accuracy_score(y_test, lr.predict(X_test))

print(f'Logistic Regression – Accuracy : {acc_lr:.4f} ({acc_lr*100:.2f}%)')

In [ ]:
# K-Nearest Neighbours
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train, y_train)
acc_knn = accuracy_score(y_test, knn.predict(X_test))

print(f'K-Nearest Neighbours (k=5) – Accuracy : {acc_knn:.4f} ({acc_knn*100:.2f}%)')

In [ ]:
# Random Forest
rf = RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE)
rf.fit(X_train, y_train)
acc_rf = accuracy_score(y_test, rf.predict(X_test))

print(f'Random Forest – Accuracy : {acc_rf:.4f} ({acc_rf*100:.2f}%)')

In [ ]:
# Support Vector Machine
svm = SVC(kernel='rbf', C=1.0, gamma='scale', probability=True, random_state=RANDOM_STATE)
svm.fit(X_train, y_train)
acc_svm = accuracy_score(y_test, svm.predict(X_test))

print(f'SVM (RBF kernel) – Accuracy : {acc_svm:.4f} ({acc_svm*100:.2f}%)')

In [ ]:
# Tableau récapitulatif des accuracies
results = {
    'Logistic Regression': acc_lr,
    'KNN (k=5)':           acc_knn,
    'Random Forest':        acc_rf,
    'SVM (RBF)':           acc_svm
}

results_df = pd.DataFrame(
    list(results.items()), columns=['Modèle', 'Accuracy']
).sort_values('Accuracy', ascending=False).reset_index(drop=True)

print('=== Comparaison des accuracies ===')
display(results_df)

best_model_name = results_df.iloc[0]['Modèle']
best_accuracy   = results_df.iloc[0]['Accuracy']
print(f'\n→ Meilleur modèle : {best_model_name} ({best_accuracy*100:.2f}%)')

In [ ]:
# Bar chart des accuracies
palette_bar = sns.color_palette('magma', n_colors=4)

plt.figure(figsize=(9, 5))
bars = plt.bar(results_df['Modèle'], results_df['Accuracy'],
               color=palette_bar, edgecolor='white')
plt.ylim(0.85, 1.02)
plt.title('Comparaison des Accuracies – 4 modèles', fontsize=13, fontweight='bold')
plt.ylabel('Accuracy')
plt.xticks(rotation=10)

for bar, val in zip(bars, results_df['Accuracy']):
    plt.text(bar.get_x() + bar.get_width() / 2, val + 0.002,
             f'{val:.3f}', ha='center', fontsize=11, fontweight='bold')

plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Matrices de confusion des 4 modèles
trained_models = {
    'Logistic Regression': lr,
    'KNN (k=5)':           knn,
    'Random Forest':        rf,
    'SVM (RBF)':           svm
}

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, (name, model) in zip(axes, trained_models.items()):
    cm = confusion_matrix(y_test, model.predict(X_test))
    ConfusionMatrixDisplay(cm, display_labels=['Bénigne', 'Maligne']).plot(
        ax=ax, cmap='magma', colorbar=False
    )
    ax.set_title(name, fontsize=9, fontweight='bold')

plt.suptitle('Matrices de Confusion – 4 modèles', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Rapport de classification détaillé du meilleur modèle
best_models_dict = {
    'Logistic Regression': lr,
    'KNN (k=5)':           knn,
    'Random Forest':        rf,
    'SVM (RBF)':           svm
}
best_model_obj = best_models_dict[best_model_name]

print(f'=== Rapport détaillé – {best_model_name} ===')
print(classification_report(
    y_test, best_model_obj.predict(X_test),
    target_names=['Bénigne (0)', 'Maligne (1)']
))

## Quel est le meilleur modèle ?

Sur ce dataset Breast Cancer Wisconsin :

- **Logistic Regression** : excellent point de départ, très interprétable, souvent ~95-97%
- **KNN (k=5)** : sensible à l'échelle (d'où la standardisation), bonnes performances ~94-96%
- **Random Forest** : capture les interactions non-linéaires entre features, souvent le meilleur ~96-98%
- **SVM (RBF)** : très efficace sur des espaces à haute dimension, souvent ~97-98%

**→ SVM et Random Forest** se disputent généralement la première place sur ce dataset.  

**Note médicale :** L'accuracy seule ne suffit pas. Dans un contexte de détection de cancer, le **Recall sur les Malignes** est crucial — manquer un cas malin (faux négatif) est beaucoup plus grave qu'une fausse alarme (faux positif).